# Đọc bảng Gold từ ClickHouse

Notebook này kết nối ClickHouse và đọc 3 bảng Gold:
- `dim_date` — bảng chiều ngày (2020–2030)
- `dim_symbol` — bảng chiều mã chứng khoán HOSE
- `fact_hose_daily_market` — bảng sự kiện OHLCV + chỉ báo kỹ thuật

## 1. Kết nối

In [1]:
import os

import clickhouse_connect
import polars as pl

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8123"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "admin")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "admin123")
CLICKHOUSE_DB = os.getenv("CLICKHOUSE_DB", "lakehouse")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DB,
)

version = client.query("SELECT version()").result_rows[0][0]
print(f"ClickHouse {version} @ {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT} / db={CLICKHOUSE_DB}")

ClickHouse 24.3.18.7 @ localhost:8123 / db=lakehouse


## 2. Danh sách bảng trong database

In [7]:
def query_to_polars(sql: str) -> pl.DataFrame:
    result = client.query(sql)
    return pl.DataFrame(
        {col: [row[i] for row in result.result_rows] for i, col in enumerate(result.column_names)}
    )


tables = query_to_polars(
    f"""
    SELECT
        name,
        engine,
        total_rows,
        formatReadableSize(total_bytes) AS size_on_disk
    FROM system.tables
    WHERE database = '{CLICKHOUSE_DB}'
      AND name NOT LIKE 'smoke%'
    ORDER BY name
    """
)
print(tables)

shape: (3, 4)
┌────────────────────────┬────────────────────┬────────────┬──────────────┐
│ name                   ┆ engine             ┆ total_rows ┆ size_on_disk │
│ ---                    ┆ ---                ┆ ---        ┆ ---          │
│ str                    ┆ str                ┆ i64        ┆ str          │
╞════════════════════════╪════════════════════╪════════════╪══════════════╡
│ dim_date               ┆ MergeTree          ┆ 4018       ┆ 27.73 KiB    │
│ dim_symbol             ┆ ReplacingMergeTree ┆ 5          ┆ 4.04 KiB     │
│ fact_hose_daily_market ┆ MergeTree          ┆ 0          ┆ 0.00 B       │
└────────────────────────┴────────────────────┴────────────┴──────────────┘


## 3. dim_date

In [8]:
dim_date = query_to_polars("SELECT * FROM dim_date ORDER BY date_key")
print(f"Rows: {len(dim_date):,}  |  Columns: {dim_date.columns}")
dim_date.head(10)

Rows: 4,018  |  Columns: ['date_key', 'full_date', 'day', 'cal_week', 'cal_month', 'cal_quarter', 'cal_year', 'is_weekend', 'event_name', 'event_type', 'is_day_off']


date_key,full_date,day,cal_week,cal_month,cal_quarter,cal_year,is_weekend,event_name,event_type,is_day_off
i64,date,i64,i64,i64,i64,i64,bool,str,str,bool
20200101,2020-01-01,1,1,1,1,2020,false,"""Tết Dương lịch""","""HOLIDAY""",true
20200102,2020-01-02,2,1,1,1,2020,false,null,null,false
20200103,2020-01-03,3,1,1,1,2020,false,null,null,false
20200104,2020-01-04,4,1,1,1,2020,true,null,null,true
20200105,2020-01-05,5,1,1,1,2020,true,null,null,true
20200106,2020-01-06,6,2,1,1,2020,false,null,null,false
20200107,2020-01-07,7,2,1,1,2020,false,null,null,false
20200108,2020-01-08,8,2,1,1,2020,false,null,null,false
20200109,2020-01-09,9,2,1,1,2020,false,null,null,false


In [4]:
# Phân bố ngày nghỉ theo năm
dim_date_stats = query_to_polars(
    """
    SELECT
        cal_year,
        countIf(is_weekend)    AS weekends,
        countIf(is_day_off)    AS day_offs,
        countIf(NOT is_day_off AND NOT is_weekend) AS trading_days
    FROM dim_date
    GROUP BY cal_year
    ORDER BY cal_year
    """
)
print(dim_date_stats)

shape: (11, 4)
┌──────────┬──────────┬──────────┬──────────────┐
│ cal_year ┆ weekends ┆ day_offs ┆ trading_days │
│ ---      ┆ ---      ┆ ---      ┆ ---          │
│ i64      ┆ i64      ┆ i64      ┆ i64          │
╞══════════╪══════════╪══════════╪══════════════╡
│ 2020     ┆ 104      ┆ 114      ┆ 252          │
│ 2021     ┆ 104      ┆ 115      ┆ 250          │
│ 2022     ┆ 105      ┆ 116      ┆ 249          │
│ 2023     ┆ 105      ┆ 116      ┆ 249          │
│ 2024     ┆ 104      ┆ 116      ┆ 250          │
│ …        ┆ …        ┆ …        ┆ …            │
│ 2026     ┆ 104      ┆ 115      ┆ 250          │
│ 2027     ┆ 104      ┆ 115      ┆ 250          │
│ 2028     ┆ 106      ┆ 117      ┆ 249          │
│ 2029     ┆ 104      ┆ 115      ┆ 250          │
│ 2030     ┆ 104      ┆ 115      ┆ 250          │
└──────────┴──────────┴──────────┴──────────────┘


## 4. dim_symbol

In [10]:
dim_symbol = query_to_polars("SELECT * FROM dim_symbol ORDER BY symbol")
print(f"Rows: {len(dim_symbol):,}  |  Columns: {dim_symbol.columns}")
dim_symbol.head(10)

Rows: 5  |  Columns: ['symbol_key', 'symbol', 'company_name', 'sector_name', 'company_profile', 'listing_date', 'exchange_code', 'listed_status', 'updated_at']


symbol_key,symbol,company_name,sector_name,company_profile,listing_date,exchange_code,listed_status,updated_at
i64,str,str,str,str,date,str,str,datetime[μs]
1,"""FPT""","""Công ty Cổ phần FPT""","""Technology""","""Công ty Cổ phần FPT (FPT) có t…",2006-12-13,"""HOSE""","""LISTED""",2026-06-15 12:27:05.621512
2,"""HPG""","""Công ty Cổ phần Tập đoàn Hòa P…","""Basic Resources""","""Công ty Cổ phần Tập đoàn Hoà P…",2007-11-15,"""HOSE""","""LISTED""",2026-06-15 12:27:05.621512
3,"""MWG""","""Công ty Cổ phần Đầu tư Thế Giớ…","""Retail""","""Công ty Cổ phần Đầu tư Thế Giớ…",2014-07-14,"""HOSE""","""LISTED""",2026-06-15 12:27:05.621512
4,"""VCB""","""Ngân hàng Thương mại Cổ phần N…","""Banks""","""Ngân hàng Thương mại Cổ phần N…",2009-06-30,"""HOSE""","""LISTED""",2026-06-15 12:27:05.621512
5,"""VNM""","""Công ty Cổ phần Sữa Việt Nam""","""Food & Beverage""","""Công ty Cổ phần Sữa Việt Nam (…",2006-01-19,"""HOSE""","""LISTED""",2026-06-15 12:27:05.621512


In [6]:
# Phân bố theo sector
sector_dist = query_to_polars(
    """
    SELECT
        coalesce(sector_name, '(unknown)') AS sector,
        count() AS symbol_count
    FROM dim_symbol
    GROUP BY sector
    ORDER BY symbol_count DESC
    LIMIT 15
    """
)
print(sector_dist)

shape: (1, 2)
┌───────────┬──────────────┐
│ sector    ┆ symbol_count │
│ ---       ┆ ---          │
│ str       ┆ i64          │
╞═══════════╪══════════════╡
│ (unknown) ┆ 1531         │
└───────────┴──────────────┘


## 5. fact_hose_daily_market

In [ ]:
# Tổng quan partition theo tháng
fact_monthly = query_to_polars(
    """
    SELECT
        toYYYYMM(trading_date) AS yyyymm,
        count()                AS rows,
        count(DISTINCT symbol_key) AS symbols
    FROM fact_hose_daily_market
    GROUP BY yyyymm
    ORDER BY yyyymm
    """
)
print(f"Partitions: {len(fact_monthly)}")
print(fact_monthly)

In [ ]:
# Đọc toàn bộ fact (nếu dữ liệu lớn, giới hạn bằng WHERE trading_date >= '2025-01-01')
fact = query_to_polars(
    """
    SELECT *
    FROM fact_hose_daily_market
    ORDER BY trading_date DESC, symbol_key
    LIMIT 50000
    """
)
print(f"Rows loaded: {len(fact):,}  |  Columns: {fact.columns}")
fact.head(10)

In [ ]:
# Top 10 mã có khối lượng giao dịch trung bình cao nhất
top_volume = query_to_polars(
    """
    SELECT
        s.symbol,
        round(avg(f.volume), 0)            AS avg_volume,
        round(avg(f.close_price), 2)       AS avg_close,
        round(avg(f.pct_change) * 100, 4)  AS avg_pct_change_pct,
        count()                            AS trading_days
    FROM fact_hose_daily_market f
    JOIN dim_symbol s ON s.symbol_key = f.symbol_key
    GROUP BY s.symbol
    ORDER BY avg_volume DESC
    LIMIT 10
    """
)
print("Top 10 mã theo khối lượng trung bình:")
print(top_volume)

In [ ]:
# Lấy lịch sử một mã cụ thể (thay 'VCB' bằng mã bạn muốn xem)
SYMBOL = "VCB"

symbol_history = query_to_polars(
    f"""
    SELECT
        f.trading_date,
        f.open_price,
        f.high_price,
        f.low_price,
        f.close_price,
        f.volume,
        f.pct_change,
        f.sma20,
        f.ema20,
        f.rsi14,
        f.macd
    FROM fact_hose_daily_market f
    JOIN dim_symbol s ON s.symbol_key = f.symbol_key
    WHERE s.symbol = '{SYMBOL}'
    ORDER BY f.trading_date
    """
)
print(f"{SYMBOL}: {len(symbol_history)} ngày giao dịch")
symbol_history.tail(20)

## 6. Join dim_date + dim_symbol + fact

In [ ]:
# Dữ liệu giao dịch tuần gần nhất (full join 3 bảng)
latest_week = query_to_polars(
    """
    SELECT
        d.full_date,
        d.cal_year,
        d.cal_month,
        s.symbol,
        s.sector_name,
        f.open_price,
        f.close_price,
        f.volume,
        f.pct_change,
        f.rsi14
    FROM fact_hose_daily_market f
    JOIN dim_date   d ON d.date_key   = f.date_key
    JOIN dim_symbol s ON s.symbol_key = f.symbol_key
    WHERE f.trading_date >= (SELECT max(trading_date) - 6 FROM fact_hose_daily_market)
    ORDER BY f.trading_date DESC, s.symbol
    """
)
print(f"Rows: {len(latest_week):,}")
latest_week.head(20)